# 01 - Data Collection: Polymarket Trade Data

**Purpose:** Fetch on-chain Polymarket trade data for both YES and NO tokens across 5 categories, with metadata for downstream slicing.

**Data sources:**
- [Gamma API](https://gamma-api.polymarket.com) -- market discovery (resolved events by category)
- [Goldsky subgraph](https://api.goldsky.com) -- on-chain trade data (`OrderFilled` events from Polymarket's CTF Exchange on Polygon)

**Sampling strategy:**
- Fetch top 50 resolved events per category by volume (no artificial token caps)
- Include ALL event types: binary, multi-outcome, negRisk
- Tag each trade with metadata (`num_markets_in_event`, `neg_risk`) so analysis notebooks can slice without re-fetching
- No filtering applied here -- filtering is a downstream analysis decision

**Outputs:**
- `../data/trades_all.parquet` -- all trades tagged with category + event metadata
- `../data/discovery_summary.json` -- per-category fetch statistics

**Runtime:** ~20-30 min first run (API fetching across ~250 events), seconds on subsequent runs (cached)

In [ ]:
import sys
from pathlib import Path

# Add project root to path so we can import from src/
sys.path.insert(0, str(Path("..").resolve()))

from src.fetch import load_or_fetch_all_categories
from src.config import CATEGORIES, DATA_DIR

## How Polymarket trades work on-chain

Every Polymarket trade is an on-chain `OrderFilled` event on the [CTF Exchange contract](https://polygonscan.com/address/0x4bFb41d5B3570DeFd03C39a9A4D8dE6Bd8B8982E) on Polygon.

**Two tokens per market:** Each market has separate ERC-1155 tokens for YES and NO. Both have their own order activity on-chain.

**Side determination from OrderFilled events:**
- `makerAssetId == "0"` (USDC) → maker **bought** tokens → `side = BUY`
- `takerAssetId == "0"` (USDC) → maker **sold** tokens → `side = SELL`

**We query both tokens:** For each market, we fetch OrderFilled events where either `makerAssetId` or `takerAssetId` matches the YES token ID, then repeat for the NO token ID. Deduplication by event ID ensures no double-counting.

**What "BUY YES" vs "BUY NO" means economically:**
- BUY YES = betting the outcome happens (bullish)
- BUY NO = betting the outcome doesn't happen (bearish)
- SELL YES = exiting a bullish position (or shorting via the order book)
- SELL NO = exiting a bearish position

See `output/polymarket-exchange-architecture.md` for the full architecture writeup.

## Fetch all categories

Discovers top 50 resolved events per category via Gamma API (sorted by volume), then fetches both YES and NO token trades from Goldsky. Each trade is tagged with:

| Field | Purpose |
|-------|---------|
| `category` | Polymarket category (Politics, Crypto, etc.) |
| `event` | Parent event title |
| `question` | Specific market question |
| `num_markets_in_event` | 1 = pure binary, 2+ = multi-outcome |
| `neg_risk` | True = mutually exclusive outcomes |
| `token_type` | YES or NO |
| `side` | BUY or SELL |

This metadata lets notebooks 02-05 slice the data any way needed without re-fetching.

In [ ]:
# Fetch all trades across 5 categories -- both YES and NO tokens
# First run: ~20-25 min (API calls). Subsequent runs: instant (cached).
# Set force_refresh=True to re-fetch from scratch.
df = load_or_fetch_all_categories(
    max_events_per_category=50,
    max_markets_per_category=100,  # cap total markets per category to keep fetch time ~25 min
    max_trades_per_token=1000,
    force_refresh=False,
)

## Data characterization

Sample overview: size, coverage, composition. This section exists so downstream
notebooks can assess whether the sample is adequate for their specific question.

In [ ]:
import pandas as pd

print(f"Total trades: {len(df):,}")
print(f"Total volume: ${df['usdc'].sum():,.0f}")
print(f"Events: {df['event'].nunique()}")
print(f"Markets (questions): {df['question'].nunique()}")

# Time range
from datetime import datetime
ts_min = datetime.fromtimestamp(df['timestamp'].min())
ts_max = datetime.fromtimestamp(df['timestamp'].max())
print(f"Time range: {ts_min:%Y-%m-%d} to {ts_max:%Y-%m-%d}")

print(f"\n--- Per category ---")
cat_summary = df.groupby("category").agg(
    trades=("timestamp", "count"),
    events=("event", "nunique"),
    markets=("question", "nunique"),
    volume=("usdc", "sum"),
).sort_values("volume", ascending=False)
cat_summary["volume"] = cat_summary["volume"].map("${:,.0f}".format)
print(cat_summary.to_string())

print(f"\n--- Token type split ---")
print(df["token_type"].value_counts().to_string())

print(f"\n--- Side split ---")
print(df["side"].value_counts().to_string())

print(f"\n--- Event structure ---")
event_sizes = df.groupby("event")["num_markets_in_event"].first()
print(f"Binary events (1 market):     {(event_sizes == 1).sum()}")
print(f"Small multi (2-4 markets):    {((event_sizes >= 2) & (event_sizes <= 4)).sum()}")
print(f"Large multi (5+ markets):     {(event_sizes >= 5).sum()}")
print(f"negRisk events:               {df.groupby('event')['neg_risk'].first().sum()}")

print(f"\n--- Token x Side crosstab ---")
print(pd.crosstab(df["token_type"], df["side"], margins=True).to_string())

## Known sampling limitations

**What this sample IS:**
- Top ~250 resolved events by volume on Polymarket, across 5 categories
- Both YES and NO token on-chain trades from the CTF Exchange
- Up to 1,000 trades per token (capped to keep fetch time reasonable)

**What this sample is NOT:**
- Not a random sample -- volume-weighted, so high-volume events dominate
- Not exhaustive -- 1,000 trade cap per token means we undersample the busiest markets
- Not all categories equally represented -- Polymarket has more Politics/Crypto than Culture
- Resolved markets only -- does not capture behavior on still-open markets

**Selection bias risks:**
- Volume sorting overweights "popular" events which may attract different trader behavior
- Multi-outcome events (elections, championships) have structurally different token price distributions than binary events
- These biases are mitigated by tagging metadata and slicing in analysis, not by filtering here

**For the next notebook:** Use `num_markets_in_event` and `neg_risk` columns to compare binary vs multi-outcome behavior. If YES bias only appears in one type, the finding is conditional, not universal.

---
## Population-level resolution data

The trade-level dataset above covers ~250 high-volume events. For resolution analysis (do single-market events resolve YES or NO?), we need the **full population** — not just the liquid tail.

This fetch pulls metadata only (no trades) for every resolved event on Polymarket via the Gamma API. It's fast (~5 min) because we only need the settlement outcome, not on-chain trade data.

**Why this matters:** Our 88 single-market events from the trade sample showed a ~51% YES resolution rate. But that sample is volume-biased. The population tells a very different story.

In [ ]:
from src.fetch import fetch_resolution_population

df_resolution = fetch_resolution_population(force_refresh=False)

print(f"Single-market events: {len(df_resolution):,}")
print(f"Resolved YES: {df_resolution['resolved_yes'].sum():,} ({df_resolution['resolved_yes'].mean()*100:.1f}%)")
print(f"Resolved NO:  {(~df_resolution['resolved_yes']).sum():,} ({(~df_resolution['resolved_yes']).mean()*100:.1f}%)")

print(f"\n--- By category ---")
cat_res = df_resolution.groupby("category").agg(
    total=("resolved_yes", "count"),
    yes=("resolved_yes", "sum"),
).assign(
    no=lambda x: x["total"] - x["yes"],
    yes_pct=lambda x: x["yes"] / x["total"] * 100,
).sort_values("total", ascending=False)
print(cat_res.to_string())